Teacher model: LLaMA-3.2-1B + LoRA
Input: Claim + Evidence + Verdict + Justification
Output: trace score
Aggregation: top-1 verdict


In [1]:
!pip install -q transformers peft evaluate tomli scikit-learn pandas tqdm torch accelerate


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# =========================
# 1. Imports
# =========================

import os
import re
import json
import sys
import shutil
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from peft import LoraConfig, get_peft_model
import re as _re


/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =========================
# 2. Paths, config & language registry
# =========================

BASE_DIR = r"/Users/valerie/Desktop/TU/AIR/AIR_Group_Task"
os.chdir(BASE_DIR)
print("Working directory:", os.getcwd())

# ── Model / training hyper-parameters (shared across languages) ──────────
BASE_MODEL   = "meta-llama/Llama-3.2-1B"
MAX_LENGTH   = 256
BATCH_SIZE   = 2
EPOCHS       = 3
LR           = 1e-4
RANDOM_STATE = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ── Language / dataset registry ──────────────────────────────────────────
# Each entry defines the raw data files and where outputs land.
# Add or remove entries here to control which languages are processed.
LANGUAGES = [
    {
        "lang"           : "english",
        "train_path"     : "data/english/english_train.json",
        "val_path"       : "data/english/clef2026_gpt4_o_mini_val.json",
        "train_jsonl"    : "output/training_data_for_RM/english_train_with_evidence.jsonl",
        "teacher_model_dir" : "output/teacher_llama_evidence_english",
        "pred_path"      : "output/RM_prediction/teacher_llama_evidence_english_predictions.json",
        "result_dir"     : "output/results_teacher_llama_evidence_english",
    },
    {
        "lang"           : "spanish",
        "train_path"     : "data/spanish/spanish_train.json",
        "val_path"       : "data/spanish/spanish_val.json",
        "train_jsonl"    : "output/training_data_for_RM/spanish_train_with_evidence.jsonl",
        "teacher_model_dir" : "output/teacher_llama_evidence_spanish",
        "pred_path"      : "output/RM_prediction/teacher_llama_evidence_spanish_predictions.json",
        "result_dir"     : "output/results_teacher_llama_evidence_spanish",
    },
    {
        "lang"           : "arabic",
        "train_path"     : "data/arabic/clef2026_gpt4_o_mini_train_arabic.json",
        "val_path"       : "data/arabic/clef2026_gpt4_o_mini_val_arabic.json",
        "train_jsonl"    : "output/training_data_for_RM/arabic_train_with_evidence.jsonl",
        "teacher_model_dir" : "output/teacher_llama_evidence_arabic",
        "pred_path"      : "output/RM_prediction/teacher_llama_evidence_arabic_predictions.json",
        "result_dir"     : "output/results_teacher_llama_evidence_arabic",
    },
]

# Create all output directories up front
os.makedirs("output/training_data_for_RM", exist_ok=True)
os.makedirs("output/RM_prediction",        exist_ok=True)
for cfg in LANGUAGES:
    os.makedirs(cfg["teacher_model_dir"], exist_ok=True)
    os.makedirs(cfg["result_dir"],        exist_ok=True)

print(f"\nConfigured {len(LANGUAGES)} language(s):")
for cfg in LANGUAGES:
    train_ok = os.path.exists(cfg["train_path"])
    val_ok   = os.path.exists(cfg["val_path"])
    print(f"  [{cfg['lang']:8s}]  train={train_ok}  val={val_ok}")


Working directory: /Users/valerie/Desktop/TU/AIR/AIR_Group_Task
Device: cpu

Configured 3 language(s):
  [english ]  train=True  val=True
  [spanish ]  train=True  val=True
  [arabic  ]  train=True  val=True


In [15]:
# =========================
# 3. Helper functions
# =========================

def remove_label_pattern(text):
    text = re.sub(
        r"\[?\s*Justification\s*\]?\s*:?",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\[?\s*Label\s*\]?\s*:\s*(True|False|Conflicting|Supports|Refutes|SUPPORTS|REFUTES|CONFLICTING)",
        "",
        text,
        flags=re.IGNORECASE,
    )

    return text.replace("\n", " ").strip()


def get_evidence(sample):
    possible_keys = [
        "evidences", "evidence", "Evidence",
        "relevant_evidence", "Relevant_evidence",
        "context", "Context",
        "gold_evidence", "Gold_evidence",
    ]
    for key in possible_keys:
        if key in sample and sample[key]:
            value = sample[key]
            if isinstance(value, list):  return " ".join(map(str, value))
            if isinstance(value, dict):  return json.dumps(value, ensure_ascii=False)
            return str(value)
    return ""


def build_teacher_input(claim, evidence, verdict, justification):
    if evidence is None:
        evidence = ""

    evidence = str(evidence).replace("\n", " ").strip()
    evidence = evidence[:700]

    return (
        f"Claim: {claim}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}\n"
        f"Evidence: {evidence}"
    )


def print_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(
        f"trainable params: {trainable} || "
        f"all params: {total} || "
        f"trainable%: {100 * trainable / total:.2f}"
    )


In [16]:
# =========================
# 4. Dataset class
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["teacher_input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item



In [17]:
# =========================
# 5. TeacherVerifier class
# =========================

class TeacherVerifier(torch.nn.Module):
    def __init__(
            self,
            model_name,
            num_labels=1,
            hidden_dim=None,
            dropout_value=0.1,
            use_lora=True,
            lora_rank=8,
            lora_alpha=16,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
        )

        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj"],
                lora_dropout=0.05,
                bias="none",
            )

            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # For decoder-style models: use last token representation
        pooled_output = outputs.last_hidden_state[:, -1, :]
        pooled_output = pooled_output.float()

        logits = self.classifier(pooled_output)

        return logits



In [18]:
# =========================
# 6. TrainerModule class
# =========================

class TrainerModule:
    def __init__(
            self,
            model,
            train_loader,
            val_loader,
            epochs,
            lr,
            output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0
            total_acc = 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (
                        torch.sigmoid(logits).squeeze(1) >= 0.5
                ).detach().cpu().numpy()

                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()

        total_loss = 0
        total_acc = 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()

                preds = (
                        torch.sigmoid(logits).squeeze(1) >= 0.5
                ).detach().cpu().numpy()

                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")

        tokenizer.save_pretrained(self.output_dir)

        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"teacher_model_epoch_{epoch}.pt"),
        )


In [19]:
# =========================
# 7. TeacherEvaluator class
# =========================

class TeacherEvaluator:
    def __init__(
            self,
            model_path,
            tokenizer_path,
            base_model,
            device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = TeacherVerifier(
            model_name=base_model,
            use_lora=True,
            lora_rank=8,
            lora_alpha=16,
        )

        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device)
        )

        self.model.to(self.device)
        self.model.eval()

    def encode_input(
            self,
            claim,
            evidence,
            verdict,
            justification,
            max_length=MAX_LENGTH,
    ):
        text = build_teacher_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, evidence, verdict, justification):
        input_ids, attention_mask = self.encode_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        with torch.no_grad():
            score = self.model(input_ids, attention_mask).item()

        return float(score)


In [9]:
# =========================
# 4. Multi-language train → predict → evaluate loop
# =========================
# Results for every language are collected in `all_results` for the
# cross-language comparison table printed at the end.

all_results = {}   # lang -> dict of metrics

for lang_cfg in LANGUAGES:
    lang             = lang_cfg["lang"]
    RAW_TRAIN_PATH   = lang_cfg["train_path"]
    VAL_PATH         = lang_cfg["val_path"]
    TRAIN_JSONL      = lang_cfg["train_jsonl"]
    TEACHER_MODEL_DIR = lang_cfg["teacher_model_dir"]
    TEACHER_PRED_PATH = lang_cfg["pred_path"]
    TEACHER_RESULT_DIR = lang_cfg["result_dir"]

    print("\n" + "=" * 60)
    print(f"  LANGUAGE: {lang.upper()}")
    print("=" * 60)

    # ── (a) Preprocessing ────────────────────────────────────────
    print(f"\n[{lang}] Step 1/4 – preprocessing training data ...")
    if os.path.exists(TRAIN_JSONL):
        print(f"[{lang}] ✓ JSONL already exists, skipping preprocessing: {TRAIN_JSONL}")
    else:
        subprocess.run(
            [
                sys.executable,
                "experiments/teacher_llama_evidence/reasoning_trace_build_with_evidence.py",
                "--input",  RAW_TRAIN_PATH,
                "--output", TRAIN_JSONL,
            ],
            check=True,
        )

    train_df = pd.read_json(TRAIN_JSONL, lines=True)
    train_df["teacher_input_text"] = train_df["input_text"]
    print(f"[{lang}] Training examples: {len(train_df)}  |  class dist:\n{train_df['Class'].value_counts().to_string()}")

    # ── (b) Train ────────────────────────────────────────────────
    # Check if all epoch checkpoints already exist — if so, skip training.
    expected_checkpoints = [
        os.path.join(TEACHER_MODEL_DIR, f"teacher_model_epoch_{e}.pt")
        for e in range(EPOCHS)
    ]
    already_trained = all(os.path.exists(p) for p in expected_checkpoints)

    print(f"\n[{lang}] Step 2/4 – training teacher verifier ({EPOCHS} epochs) ...")
    if already_trained:
        print(f"[{lang}] ✓ All {EPOCHS} checkpoints found, skipping training:")
        for p in expected_checkpoints:
            size_gb = os.path.getsize(p) / (1024 ** 3)
            print(f"         {os.path.basename(p)}  ({size_gb:.2f} GB)")
    else:
        missing = [p for p in expected_checkpoints if not os.path.exists(p)]
        print(f"[{lang}] {len(missing)} checkpoint(s) missing — running training ...")

        train_split, dev_split = train_test_split(
            train_df,
            test_size=0.2,
            stratify=train_df["Class"],
            random_state=RANDOM_STATE,
        )

        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        train_dataset = TextDataset(train_split, tokenizer, MAX_LENGTH)
        dev_dataset   = TextDataset(dev_split,   tokenizer, MAX_LENGTH)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE)

        teacher_model = TeacherVerifier(
            model_name=BASE_MODEL,
            use_lora=True,
            lora_rank=8,
            lora_alpha=16,
        )
        print_trainable_parameters(teacher_model)

        trainer = TrainerModule(
            model=teacher_model,
            train_loader=train_loader,
            val_loader=dev_loader,
            epochs=EPOCHS,
            lr=LR,
            output_dir=TEACHER_MODEL_DIR,
        )
        trainer.train()

    # ── (c) Generate predictions ─────────────────────────────────
    print(f"\n[{lang}] Step 3/4 – generating teacher predictions ...")

    BEST_EPOCH          = EPOCHS - 1
    TEACHER_MODEL_PATH  = os.path.join(TEACHER_MODEL_DIR, f"teacher_model_epoch_{BEST_EPOCH}.pt")

    with open(VAL_PATH, "r", encoding="utf-8") as f:
        val_data = json.load(f)

    teacher_evaluator = TeacherEvaluator(
        model_path=TEACHER_MODEL_PATH,
        tokenizer_path=TEACHER_MODEL_DIR,
        base_model=BASE_MODEL,
    )

    predictions = []
    for idx, sample in enumerate(tqdm(val_data, desc=f"{lang} inference")):
        claim    = sample["claim"]
        evidence = get_evidence(sample)

        verdict_list         = []
        verifier_score_list  = []
        justification_list   = []

        for trace_idx in range(len(sample["Reasoning_traces"])):
            justification = remove_label_pattern(
                sample["Reasoning_traces"][trace_idx]
            ).split("Label:")[0]
            verdict = sample["Verdict_list"][trace_idx].lower()

            score = teacher_evaluator.score(
                claim=claim, evidence=evidence,
                verdict=verdict, justification=justification,
            )
            verdict_list.append(sample["Verdict_list"][trace_idx])
            justification_list.append(justification)
            verifier_score_list.append(score)

        best_idx    = int(np.argmax(np.array(verifier_score_list)))
        best_verdict = verdict_list[best_idx]

        predictions.append({
            "query_id"       : sample.get("query_id", idx),
            "Claim"          : claim,
            "Evidence"       : evidence,
            "Label"          : sample["label"],
            "Verdict_BoN"    : best_verdict,
            "BoN_Verdict_list": verdict_list,
            "Reasoning_traces": justification_list,
            "score_list"     : verifier_score_list,
        })

    with open(TEACHER_PRED_PATH, "w", encoding="utf-8") as fp:
        json.dump(predictions, fp, indent=4, ensure_ascii=False)
    print(f"[{lang}] Saved {len(predictions)} predictions → {TEACHER_PRED_PATH}")

    # ── (d) Score ────────────────────────────────────────────────
    print(f"\n[{lang}] Step 4/4 – running scorer ...")

    shutil.copy(TEACHER_PRED_PATH, "output/RM_prediction/clef_predictions.json")
    subprocess.run([sys.executable, "task2/scorer.py"], check=True)

    shutil.copy("output/RM_prediction/result.csv",       f"{TEACHER_RESULT_DIR}/result.csv")
    shutil.copy("output/RM_prediction/per_sample_ir.csv", f"{TEACHER_RESULT_DIR}/per_sample_ir.csv")

    # ── collect summary metrics ───────────────────────────────────
    with open(f"{TEACHER_RESULT_DIR}/result.csv", "r") as f:
        content = f.read()
    m_f1 = _re.search(r"macro avg,[0-9.]+,[0-9.]+,([0-9.]+)", content)
    m_r5 = _re.search(r"^\s*5,([0-9.]+)", content, _re.MULTILINE)
    macro_f1   = float(m_f1.group(1)) if m_f1 else float("nan")
    recall_at5 = float(m_r5.group(1)) if m_r5 else float("nan")

    all_results[lang] = {
        "macro_f1"  : macro_f1,
        "recall@5"  : recall_at5,
        "n_val"     : len(predictions),
        "n_train"   : len(train_df),
    }
    print(f"[{lang}] Done. macro F1={macro_f1:.4f}  Recall@5={recall_at5:.4f}")

# ── cross-language summary table ─────────────────────────────────────────
print("\n" + "=" * 60)
print("CROSS-LANGUAGE RESULTS SUMMARY")
print("=" * 60)
print(f"{'Language':<12} {'Train':>8} {'Val':>6} {'Macro F1':>10} {'Recall@5':>10}")
print("-" * 60)
for lang, m in all_results.items():
    print(f"{lang:<12} {m['n_train']:>8,} {m['n_val']:>6,} {m['macro_f1']:>10.4f} {m['recall@5']:>10.4f}")
print("=" * 60)



  LANGUAGE: ENGLISH

[english] Step 1/4 – preprocessing training data ...
[english] ✓ JSONL already exists, skipping preprocessing: output/training_data_for_RM/english_train_with_evidence.jsonl
[english] Training examples: 31433  |  class dist:
Class
0    22775
1     8658

[english] Step 2/4 – training teacher verifier (3 epochs) ...
[english] ✓ All 3 checkpoints found, skipping training:
         teacher_model_epoch_0.pt  (4.61 GB)
         teacher_model_epoch_1.pt  (4.61 GB)
         teacher_model_epoch_2.pt  (4.61 GB)

[english] Step 3/4 – generating teacher predictions ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
english inference: 100%|██████████| 1600/1600 [3:43:12<00:00,  8.37s/it]  


[english] Saved 1600 predictions → output/RM_prediction/teacher_llama_evidence_english_predictions.json

[english] Step 4/4 – running scorer ...
Total unique claims: 1600

--- IR Ranking Metrics ---
    k   Mean Recall@k  Mean Precision@k
    1          0.0808            0.5625
    2          0.1418            0.5569
    3          0.1952            0.5504
    4          0.2456            0.5458
    5          0.2937            0.5417

--- Per-Class Metrics (Verdict_BoN) ---
         false:  P=0.7891  R=0.6188  F1=0.6937  n=913
          true:  P=0.3619  R=0.5132  F1=0.4245  n=304
   conflicting:  P=0.3951  R=0.4674  F1=0.4282  n=383

--- Aggregate Metrics ---
  macro avg:  P=0.5154  R=0.5331  F1=0.5155
  weighted avg:  P=0.6136  R=0.5625  F1=0.5790

Aggregate results  -> output/RM_prediction/result.csv
Per-sample results -> output/RM_prediction/per_sample_ir.csv
[english] Done. macro F1=0.5155  Recall@5=0.2937

  LANGUAGE: SPANISH

[spanish] Step 1/4 – preprocessing training data ...


spanish inference: 100%|██████████| 562/562 [1:40:39<00:00, 10.75s/it]


[spanish] Saved 562 predictions → output/RM_prediction/teacher_llama_evidence_spanish_predictions.json

[spanish] Step 4/4 – running scorer ...
Total unique claims: 562

--- IR Ranking Metrics ---
    k   Mean Recall@k  Mean Precision@k
    1          0.0394            0.6246
    2          0.0835            0.6290
    3          0.1207            0.6317
    4          0.1600            0.6303
    5          0.2017            0.6306

--- Per-Class Metrics (Verdict_BoN) ---
         false:  P=0.8841  R=0.6934  F1=0.7773  n=473
          true:  P=0.0920  R=0.2105  F1=0.1280  n=38
   conflicting:  P=0.1442  R=0.2941  F1=0.1935  n=51

--- Aggregate Metrics ---
  macro avg:  P=0.3734  R=0.3994  F1=0.3663
  weighted avg:  P=0.7634  R=0.6246  F1=0.6804

Aggregate results  -> output/RM_prediction/result.csv
Per-sample results -> output/RM_prediction/per_sample_ir.csv
[spanish] Done. macro F1=0.3663  Recall@5=0.2017

  LANGUAGE: ARABIC

[arabic] Step 1/4 – preprocessing training data ...
[arabi

...

Train Loss: 0.5555
Train Acc: 0.7129
Val Loss: 0.5691
Val Acc: 0.6889

Epoch 3/3


100%|██████████| 4042/4042 [2:26:28<00:00,  2.17s/it]  


Train Loss: 0.4172
Train Acc: 0.8012
Val Loss: 0.5548
Val Acc: 0.7047

[arabic] Step 3/4 – generating teacher predictions ...


arabic inference: 100%|██████████| 652/652 [1:54:24<00:00, 10.53s/it]


[arabic] Saved 652 predictions → output/RM_prediction/teacher_llama_evidence_arabic_predictions.json

[arabic] Step 4/4 – running scorer ...
Total unique claims: 644

--- IR Ranking Metrics ---
    k   Mean Recall@k  Mean Precision@k
    1          0.0425            0.6941
    2          0.0813            0.6918
    3          0.1220            0.6951
    4          0.1701            0.6991
    5          0.2103            0.7000

--- Per-Class Metrics (Verdict_BoN) ---
         false:  P=0.8148  R=0.6836  F1=0.7435  n=354
          true:  P=0.7621  R=0.7069  F1=0.7335  n=290
   conflicting:  P=0.0000  R=0.0000  F1=0.0000  n=0

--- Aggregate Metrics ---
  macro avg:  P=0.5256  R=0.4635  F1=0.4923
  weighted avg:  P=0.7911  R=0.6941  F1=0.7390

Aggregate results  -> output/RM_prediction/result.csv
Per-sample results -> output/RM_prediction/per_sample_ir.csv
[arabic] Done. macro F1=0.4923  Recall@5=0.2103

CROSS-LANGUAGE RESULTS SUMMARY
Language        Train    Val   Macro F1   Recall@5


In [10]:
with open(f"{TEACHER_RESULT_DIR}/result.csv", "r") as f:
        content = f.read()
m_f1 = _re.search(r"macro avg,[0-9.]+,[0-9.]+,([0-9.]+)", content)
m_r5 = _re.search(r"^\s*5,([0-9.]+)", content, _re.MULTILINE)
macro_f1   = float(m_f1.group(1)) if m_f1 else float("nan")
recall_at5 = float(m_r5.group(1)) if m_r5 else float("nan")

all_results[lang] = {
        "macro_f1"  : macro_f1,
        "recall@5"  : recall_at5,
        "n_val"     : len(predictions),
        "n_train"   : len(train_df),
    }
print(f"[{lang}] Done. macro F1={macro_f1:.4f}  Recall@5={recall_at5:.4f}")

# ── cross-language summary table ─────────────────────────────────────────
print("\n" + "=" * 60)
print("CROSS-LANGUAGE RESULTS SUMMARY")
print("=" * 60)
print(f"{'Language':<12} {'Train':>8} {'Val':>6} {'Macro F1':>10} {'Recall@5':>10}")
print("-" * 60)
for lang, m in all_results.items():
    print(f"{lang:<12} {m['n_train']:>8,} {m['n_val']:>6,} {m['macro_f1']:>10.4f} {m['recall@5']:>10.4f}")
print("=" * 60)

[arabic] Done. macro F1=0.4923  Recall@5=0.2103

CROSS-LANGUAGE RESULTS SUMMARY
Language        Train    Val   Macro F1   Recall@5
------------------------------------------------------------
english        31,433  1,600     0.5155     0.2937
spanish         9,038    562     0.3663     0.2017
arabic         10,105    652     0.4923     0.2103


In [11]:
# =========================
# 5. Inference Latency & Efficiency
#    Measures real per-trace inference time per language.
#    Also applies all aggregation strategies to existing predictions
#    and reports latency per (language, aggregation) combination.
# =========================

import time
from collections import Counter

N_WARMUP      = 3    # warm-up forward passes (discarded)
N_BENCH_TRACES = 50  # number of real traces to time per language

# ── Aggregation strategies (same as RoBERTa notebook for comparison) ─────
def agg_top1(verdict_list, score_list):
    return verdict_list[int(np.argmax(score_list))]

def agg_majority_top3(verdict_list, score_list):
    top_idx = np.argsort(score_list)[::-1][:min(3, len(verdict_list))]
    return Counter([verdict_list[i] for i in top_idx]).most_common(1)[0][0]

def agg_majority_top5(verdict_list, score_list):
    top_idx = np.argsort(score_list)[::-1][:min(5, len(verdict_list))]
    return Counter([verdict_list[i] for i in top_idx]).most_common(1)[0][0]

def agg_score_weighted(verdict_list, score_list):
    weights = torch.sigmoid(torch.tensor(score_list)).numpy()
    tally = {}
    for v, w in zip(verdict_list, weights):
        tally[v] = tally.get(v, 0.0) + float(w)
    return max(tally, key=tally.get)

AGGREGATIONS = [
    ("top1",           agg_top1),
    ("majority_top3",  agg_majority_top3),
    ("majority_top5",  agg_majority_top5),
    ("score_weighted", agg_score_weighted),
]

# ── Model size (load once from any checkpoint to get param counts) ────────
def get_model_size_mb(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, total * 4 / (1024 ** 2)

_first_ckpt = os.path.join(LANGUAGES[0]["teacher_model_dir"], f"teacher_model_epoch_{EPOCHS-1}.pt")
_tmp_model  = TeacherEvaluator(
    model_path=_first_ckpt,
    tokenizer_path=LANGUAGES[0]["teacher_model_dir"],
    base_model=BASE_MODEL,
)
total_params, trainable_params, param_size_mb = get_model_size_mb(_tmp_model.model)
lora_ratio = trainable_params / total_params * 100
del _tmp_model

print("=" * 60)
print("MODEL SIZE  (LLaMA-3.2-1B + LoRA, shared across languages)")
print("=" * 60)
print(f"  Total parameters       : {total_params:,}")
print(f"  Trainable (LoRA) params: {trainable_params:,}  ({lora_ratio:.2f}%)")
print(f"  Frozen parameters      : {total_params - trainable_params:,}")
print(f"  Param memory (float32) : {param_size_mb:.0f} MB")
for lc in LANGUAGES:
    ckpt = os.path.join(lc["teacher_model_dir"], f"teacher_model_epoch_{EPOCHS-1}.pt")
    if os.path.exists(ckpt):
        gb = os.path.getsize(ckpt) / (1024**3)
        print(f"  Checkpoint ({lc['lang']:8s})  : {gb:.2f} GB")

# ── Per-language inference latency ────────────────────────────────────────
# Loads each language's checkpoint, runs N_BENCH_TRACES real traces,
# records per-trace latency, then computes aggregation overhead.

latency_records = []   # one dict per (lang, aggregation)
_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for lang_cfg in LANGUAGES:
    lang      = lang_cfg["lang"]
    ckpt_path = os.path.join(lang_cfg["teacher_model_dir"], f"teacher_model_epoch_{EPOCHS-1}.pt")
    val_path  = lang_cfg["val_path"]

    if not os.path.exists(ckpt_path):
        print(f"[{lang}] ✗ checkpoint not found, skipping latency benchmark")
        continue
    if not os.path.exists(val_path):
        print(f"[{lang}] ✗ val file not found, skipping latency benchmark")
        continue

    print(f"\n[{lang}] Loading model for latency benchmark ...")
    evaluator = TeacherEvaluator(
        model_path=ckpt_path,
        tokenizer_path=lang_cfg["teacher_model_dir"],
        base_model=BASE_MODEL,
    )
    evaluator.model.to(_device)
    evaluator.model.eval()

    with open(val_path, "r", encoding="utf-8") as f:
        val_data = json.load(f)

    # collect up to N_BENCH_TRACES individual traces across samples
    bench_inputs = []
    for sample in val_data:
        claim    = sample["claim"]
        evidence = get_evidence(sample)
        for trace_idx in range(len(sample["Reasoning_traces"])):
            justification = remove_label_pattern(
                sample["Reasoning_traces"][trace_idx]
            ).split("Label:")[0]
            verdict = sample["Verdict_list"][trace_idx].lower()
            bench_inputs.append((claim, evidence, verdict, justification))
            if len(bench_inputs) >= N_WARMUP + N_BENCH_TRACES:
                break
        if len(bench_inputs) >= N_WARMUP + N_BENCH_TRACES:
            break

    # warm-up
    for claim, evidence, verdict, justification in bench_inputs[:N_WARMUP]:
        _ = evaluator.score(claim, evidence, verdict, justification)

    # timed runs — one score() call = one trace forward pass
    trace_latencies = []
    for claim, evidence, verdict, justification in bench_inputs[N_WARMUP:]:
        if _device.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = evaluator.score(claim, evidence, verdict, justification)
        if _device.type == "cuda": torch.cuda.synchronize()
        trace_latencies.append((time.perf_counter() - t0) * 1000)

    trace_lat = np.array(trace_latencies)
    mean_trace_ms = trace_lat.mean()
    print(f"[{lang}] mean per-trace latency: {mean_trace_ms:.1f} ms  "
          f"(p50={np.percentile(trace_lat,50):.1f}  "
          f"p95={np.percentile(trace_lat,95):.1f})")

    # ── per-aggregation latency ────────────────────────────────────────────
    # Load saved predictions to get real score_lists per sample
    pred_path = lang_cfg["pred_path"]
    if not os.path.exists(pred_path):
        print(f"[{lang}] ✗ predictions file not found, skipping aggregation timing")
        del evaluator
        continue

    with open(pred_path, "r", encoding="utf-8") as f:
        predictions = json.load(f)

    # sample up to 200 predictions for aggregation timing
    sample_preds = predictions[:200]

    for agg_name, agg_fn in AGGREGATIONS:
        agg_times = []
        for pred in sample_preds:
            t0 = time.perf_counter()
            _ = agg_fn(pred["BoN_Verdict_list"], pred["score_list"])
            agg_times.append((time.perf_counter() - t0) * 1000)

        mean_agg_ms = np.mean(agg_times)
        # total = scoring all traces + aggregation
        # approximate traces per sample from the first prediction
        avg_traces = np.mean([len(p["score_list"]) for p in sample_preds])
        total_per_sample_ms = mean_trace_ms * avg_traces + mean_agg_ms

        latency_records.append({
            "model"              : "LLaMA-1B-LoRA",
            "lang"               : lang,
            "aggregation"        : agg_name,
            "mean_trace_ms"      : round(mean_trace_ms, 2),
            "avg_traces_per_sample": round(avg_traces, 1),
            "mean_agg_ms"        : round(mean_agg_ms, 4),
            "total_per_sample_ms": round(total_per_sample_ms, 2),
            "throughput_samp_sec": round(1000 / total_per_sample_ms, 3),
        })

    del evaluator

# ── Print latency comparison table ────────────────────────────────────────
lat_df = pd.DataFrame(latency_records)

print("\n" + "=" * 80)
print("INFERENCE LATENCY  —  LLaMA-1B-LoRA")
print(f"  Device: {_device}  |  Timed on {N_BENCH_TRACES} real traces (warm-up={N_WARMUP})")
print("=" * 80)
print(f"{'Language':<10} {'Aggregation':<16} {'ms/trace':>9} {'traces/samp':>12} "
      f"{'agg_ms':>8} {'ms/sample':>10} {'samp/sec':>9}")
print("-" * 80)
for _, row in lat_df.sort_values(["lang", "aggregation"]).iterrows():
    print(f"{row['lang']:<10} {row['aggregation']:<16} "
          f"{row['mean_trace_ms']:>9.1f} {row['avg_traces_per_sample']:>12.1f} "
          f"{row['mean_agg_ms']:>8.3f} {row['total_per_sample_ms']:>10.1f} "
          f"{row['throughput_samp_sec']:>9.3f}")
print("=" * 80)

# ── Performance vs efficiency combined ────────────────────────────────────
print("\n" + "=" * 80)
print("PERFORMANCE vs. EFFICIENCY  (top-1 aggregation)")
print("=" * 80)
print(f"{'Language':<10} {'Macro F1':>10} {'Recall@5':>10} {'ms/sample':>10} {'samp/sec':>9}")
print("-" * 80)
top1_df = lat_df[lat_df["aggregation"] == "top1"]
for _, row in top1_df.iterrows():
    m = all_results.get(row["lang"], {})
    print(f"{row['lang']:<10} "
          f"{m.get('macro_f1', float('nan')):>10.4f} "
          f"{m.get('recall@5', float('nan')):>10.4f} "
          f"{row['total_per_sample_ms']:>10.1f} "
          f"{row['throughput_samp_sec']:>9.3f}")
print("=" * 80)
print(f"  Model size : {param_size_mb:.0f} MB  (float32)")
print(f"  LoRA ratio : {lora_ratio:.2f}%  ({trainable_params:,} / {total_params:,} trainable params)")

# save latency table
lat_out = os.path.join(BASE_DIR, "output", "llama_latency.csv")
lat_df.to_csv(lat_out, index=False)
print(f"\nLatency table saved to: {lat_out}")


Loading weights: 100%|██████████| 146/146 [00:04<00:00, 30.99it/s]


MODEL SIZE  (LLaMA-3.2-1B + LoRA, shared across languages)
  Total parameters       : 1,236,996,097
  Trainable (LoRA) params: 1,181,697  (0.10%)
  Frozen parameters      : 1,235,814,400
  Param memory (float32) : 4719 MB
  Checkpoint (english )  : 4.61 GB
  Checkpoint (spanish )  : 4.61 GB
  Checkpoint (arabic  )  : 4.61 GB

[english] Loading model for latency benchmark ...


Loading weights: 100%|██████████| 146/146 [00:00<00:00, 146.68it/s]


[english] mean per-trace latency: 478.5 ms  (p50=472.6  p95=485.2)

[spanish] Loading model for latency benchmark ...


Loading weights: 100%|██████████| 146/146 [00:01<00:00, 137.92it/s]


[spanish] mean per-trace latency: 479.7 ms  (p50=472.0  p95=492.1)

[arabic] Loading model for latency benchmark ...


Loading weights: 100%|██████████| 146/146 [00:01<00:00, 124.14it/s]


[arabic] mean per-trace latency: 474.0 ms  (p50=469.1  p95=485.4)

INFERENCE LATENCY  —  LLaMA-1B-LoRA
  Device: cpu  |  Timed on 50 real traces (warm-up=3)
Language   Aggregation       ms/trace  traces/samp   agg_ms  ms/sample  samp/sec
--------------------------------------------------------------------------------
arabic     majority_top3        474.0         20.0    0.004     9480.9     0.105
arabic     majority_top5        474.0         20.0    0.003     9480.9     0.105
arabic     score_weighted       474.0         20.0    0.006     9480.9     0.105
arabic     top1                 474.0         20.0    0.002     9480.9     0.105
english    majority_top3        478.5         15.0    0.004     7177.3     0.139
english    majority_top5        478.5         15.0    0.003     7177.3     0.139
english    score_weighted       478.5         15.0    0.009     7177.3     0.139
english    top1                 478.5         15.0    0.002     7177.3     0.139
spanish    majority_top3        4

In [12]:
# =========================
# 6. Prepare best runs for CLEF submission (LLaMA teacher)
# =========================

import json
import shutil
import os

RUNS_DIR = os.path.join(BASE_DIR, "runs")
os.makedirs(RUNS_DIR, exist_ok=True)

# ── Convert predictions JSON → TREC format ───────────────────────────────
# TREC format: query_id  Q0  trace_id  rank  score  run_tag
def convert_to_trec(pred_path, trec_path, run_tag):
    with open(pred_path, "r", encoding="utf-8") as f:
        predictions = json.load(f)

    with open(trec_path, "w", encoding="utf-8") as out:
        for sample in predictions:
            query_id = sample["query_id"]
            scores   = sample["score_list"]
            ranked   = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
            for rank, (trace_idx, score) in enumerate(ranked, start=1):
                trace_id = f"{query_id}_{trace_idx}"
                out.write(f"{query_id}\tQ0\t{trace_id}\t{rank}\t{score:.6f}\t{run_tag}\n")

    print(f"  Written {len(predictions)} queries → {trec_path}")

# ── Pick best config per language and export ──────────────────────────────
print("=" * 60)
print("BEST RUNS PER LANGUAGE  (LLaMA teacher)")
print("=" * 60)

for lang, m in all_results.items():
    pred_path = LANGUAGES[[l["lang"] for l in LANGUAGES].index(lang)]["pred_path"]

    run_tag       = f"llama_teacher_{lang}"
    trec_filename = f"run_llama_{lang}.txt"
    trec_path     = os.path.join(RUNS_DIR, trec_filename)

    print(f"\n  [{lang.upper()}]")
    print(f"    Macro F1 : {m['macro_f1']:.4f}")
    print(f"    Recall@5 : {m['recall@5']:.4f}")

    if not os.path.exists(pred_path):
        print(f"    ✗ ERROR: prediction file not found!")
        continue

    # copy raw JSON into runs/ (for repo)
    shutil.copy(pred_path, os.path.join(RUNS_DIR, f"run_llama_{lang}.json"))

    # convert to TREC and save
    convert_to_trec(pred_path, trec_path, run_tag)
    print(f"    TREC file: {trec_path}")

BEST RUNS PER LANGUAGE  (LLaMA teacher)

  [ENGLISH]
    Macro F1 : 0.5155
    Recall@5 : 0.2937
  Written 1600 queries → /Users/valerie/Desktop/TU/AIR/AIR_Group_Task/runs/run_llama_english.txt
    TREC file: /Users/valerie/Desktop/TU/AIR/AIR_Group_Task/runs/run_llama_english.txt

  [SPANISH]
    Macro F1 : 0.3663
    Recall@5 : 0.2017
  Written 562 queries → /Users/valerie/Desktop/TU/AIR/AIR_Group_Task/runs/run_llama_spanish.txt
    TREC file: /Users/valerie/Desktop/TU/AIR/AIR_Group_Task/runs/run_llama_spanish.txt

  [ARABIC]
    Macro F1 : 0.4923
    Recall@5 : 0.2103
  Written 652 queries → /Users/valerie/Desktop/TU/AIR/AIR_Group_Task/runs/run_llama_arabic.txt
    TREC file: /Users/valerie/Desktop/TU/AIR/AIR_Group_Task/runs/run_llama_arabic.txt


In [14]:
# =========================
# 7. Test set predictions for CodaBench submission
#    Runs the trained model on each language's test file
#    and saves a JSON in the required CodaBench format.
# =========================

# ── Test file registry ────────────────────────────────────────────────────
TEST_FILES = [
    {
        "lang"        : "english",
        "test_path"   : "data/english/clef_2026_final_english_test.json",
        "model_dir"   : "output/teacher_llama_evidence_english",
        "output_path" : "codabench_submissions/test_llama_english_predictions.json",
    },
    {
        "lang"        : "spanish",
        "test_path"   : "data/spanish/clef_spanish_test_final.json",
        "model_dir"   : "output/teacher_llama_evidence_spanish",
        "output_path" : "codabench_submissions/test_llama_spanish_predictions.json",
    },
    {
        "lang"        : "arabic",
        "test_path"   : "data/arabic/clef_2026_final_arabic_test.json",
        "model_dir"   : "output/teacher_llama_evidence_arabic",
        "output_path" : "codabench_submissions/test_llama_arabic_predictions.json",
    },
]

# ── Aggregation to use on test set ────────────────────────────────────────
# Change to agg_majority_top3 / agg_majority_top5 / agg_score_weighted
# based on your val set results.
TEST_AGG_FN   = agg_top1
TEST_AGG_NAME = "top1"

os.makedirs("output/RM_prediction", exist_ok=True)
os.makedirs("codabench_submissions", exist_ok=True)

for test_cfg in TEST_FILES:
    lang      = test_cfg["lang"]
    test_path = test_cfg["test_path"]
    model_dir = test_cfg["model_dir"]
    out_path  = test_cfg["output_path"]
    ckpt_path = os.path.join(model_dir, f"teacher_model_epoch_{EPOCHS - 1}.pt")

    print(f"\n{'='*60}")
    print(f"  TEST INFERENCE: {lang.upper()}")
    print(f"{'='*60}")

    if not os.path.exists(test_path):
        print(f"  ✗ Test file not found: {test_path}")
        continue
    if not os.path.exists(ckpt_path):
        print(f"  ✗ Checkpoint not found: {ckpt_path}")
        continue

    with open(test_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)
    print(f"  Loaded {len(test_data):,} test samples")

    evaluator = TeacherEvaluator(
        model_path=ckpt_path,
        tokenizer_path=model_dir,
        base_model=BASE_MODEL,
    )

    predictions = []
    for idx, sample in enumerate(tqdm(test_data, desc=f"{lang} test")):
        # Spanish uses query_index, others use query_id
        query_id = sample.get("query_id", sample.get("query_index", idx))
        claim    = sample["claim"]
        evidence = get_evidence(sample)

        verdict_list       = []
        score_list         = []
        justification_list = []

        for trace_idx in range(len(sample["Reasoning_traces"])):
            justification = remove_label_pattern(
                sample["Reasoning_traces"][trace_idx]
            ).split("Label:")[0]
            verdict = sample["Verdict_list"][trace_idx].lower()

            score = evaluator.score(
                claim=claim, evidence=evidence,
                verdict=verdict, justification=justification,
            )
            verdict_list.append(sample["Verdict_list"][trace_idx])
            justification_list.append(justification)
            score_list.append(score)

        best_verdict = TEST_AGG_FN(verdict_list, score_list)

        predictions.append({
            "query_id"        : query_id,
            "Claim"           : claim,
            "Label"           : sample.get("label", ""),  # empty for test
            "Verdict_BoN"     : best_verdict,
            "BoN_Verdict_list": verdict_list,
            "Reasoning_traces": justification_list,
            "score_list"      : score_list,
        })

    with open(out_path, "w", encoding="utf-8") as fp:
        json.dump(predictions, fp, indent=4, ensure_ascii=False)

    print(f"  ✓ Saved {len(predictions):,} predictions → {out_path}")
    del evaluator

print("\n" + "=" * 60)
print("TEST PREDICTION FILES")
print("=" * 60)
for test_cfg in TEST_FILES:
    path = test_cfg["output_path"]
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"  ✓ {test_cfg['lang']:8s}  {path}  ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ {test_cfg['lang']:8s}  NOT FOUND")


  TEST INFERENCE: ENGLISH
  Loaded 2,558 test samples


english test: 100%|██████████| 2558/2558 [7:41:15<00:00, 10.82s/it]  


  ✓ Saved 2,558 predictions → codabench_submissions/test_llama_english_predictions.json

  TEST INFERENCE: SPANISH
  Loaded 1,164 test samples


spanish test: 100%|██████████| 1164/1164 [3:25:27<00:00, 10.59s/it] 


  ✓ Saved 1,164 predictions → codabench_submissions/test_llama_spanish_predictions.json

  TEST INFERENCE: ARABIC
  Loaded 511 test samples


arabic test: 100%|██████████| 511/511 [1:25:49<00:00, 10.08s/it]


  ✓ Saved 511 predictions → codabench_submissions/test_llama_arabic_predictions.json

TEST PREDICTION FILES
  ✓ english   codabench_submissions/test_llama_english_predictions.json  (38.4 MB)
  ✓ spanish   codabench_submissions/test_llama_spanish_predictions.json  (14.1 MB)
  ✓ arabic    codabench_submissions/test_llama_arabic_predictions.json  (5.8 MB)
